# Advanced SQL / Data Engineering Questions with Solutions

## Daily Active Users (DAU)

### Scenario
Find unique active users per day.

### Solution

In [ ]:
SELECT activity_date,
       COUNT(DISTINCT user_id) AS active_users
FROM user_activity
GROUP BY activity_date
ORDER BY activity_date;

### Result
| activity_date | active_users |
|---|---|
| 2024-01-01 | 2 |
| 2024-01-02 | 2 |

## Retention Rate

### Scenario
Find users who returned the next day.

### Solution

In [ ]:
SELECT COUNT(DISTINCT a.user_id) AS retained_users
FROM user_activity a
JOIN user_activity b
ON a.user_id = b.user_id
AND b.activity_date = DATE_ADD(a.activity_date, INTERVAL 1 DAY);

### Result
| retained_users |
|---|
| 1 |

## Median Salary

### Scenario
Find median salary from employees table.

### Solution

In [ ]:
SELECT salary AS median_salary
FROM (
    SELECT salary,
           ROW_NUMBER() OVER(ORDER BY salary) AS rn,
           COUNT(*) OVER() AS total_rows
    FROM employees
) t
WHERE rn = (total_rows + 1) / 2;

### Result
| median_salary |
|---|
| 60000 |

## Consecutive Login Days

### Scenario
Find previous login date using LAG().

### Solution

In [ ]:
SELECT user_id,
       login_date,
       LAG(login_date) OVER(
           PARTITION BY user_id
           ORDER BY login_date
       ) AS previous_login
FROM logins;

### Result
| user_id | login_date | previous_login |
|---|---|---|
| 1 | 2024-01-02 | 2024-01-01 |

## Remove Duplicate Records Keeping Latest Entry

### Scenario
Keep latest employee record using ROW_NUMBER().

### Solution

In [ ]:
SELECT emp_id,
       updated_at
FROM (
    SELECT *,
           ROW_NUMBER() OVER(
               PARTITION BY emp_id
               ORDER BY updated_at DESC
           ) AS r
    FROM employee_logs
) t
WHERE r = 1;

### Result
| emp_id | updated_at |
|---|---|
| 1 | 2024-01-05 |
| 2 | 2024-01-02 |

## Find Gaps in Dates

### Scenario
Identify missing date ranges using LAG().

### Solution

In [ ]:
SELECT attendance_date,
       LAG(attendance_date) OVER(
           ORDER BY attendance_date
       ) AS previous_date
FROM attendance;

### Result
Gap exists between 2024-01-02 and 2024-01-05

## Rolling 7-Day Average

### Scenario
Calculate rolling average sales.

### Solution

In [ ]:
SELECT sale_date,
       amount,
       AVG(amount) OVER(
           ORDER BY sale_date
           ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
       ) AS rolling_7day_avg
FROM sales;

### Result
| sale_date | amount | rolling_7day_avg |
|---|---|---|
| 2024-01-04 | 400 | 250 |

## Top-Selling Product Per Category

### Scenario
Find highest selling product per category.

### Solution

In [ ]:
SELECT category,
       product,
       sales
FROM (
    SELECT *,
           ROW_NUMBER() OVER(
               PARTITION BY category
               ORDER BY sales DESC
           ) AS r
    FROM sales
) t
WHERE r = 1;

### Result
| category | product | sales |
|---|---|---|
| Electronics | Mobile | 1500 |

## Detect Fraudulent Duplicate Transactions

### Scenario
Find suspicious duplicate transactions.

### Solution

In [ ]:
SELECT *
FROM (
    SELECT *,
           LAG(amount) OVER(
               PARTITION BY user_id
               ORDER BY transaction_time
           ) AS previous_amount
    FROM transactions
) t
WHERE amount = previous_amount;

### Result
| transaction_id | user_id | amount |
|---|---|---|
| 2 | 101 | 500 |

## SCD Type 1 vs Type 2

### Scenario
Handle slowly changing dimensions in data warehouse.

### Solution

In [ ]:
-- Expire old row
UPDATE customers
SET end_date = CURRENT_DATE,
    is_current = 'N'
WHERE customer_id = 1
AND is_current = 'Y';

-- Insert new row
INSERT INTO customers
VALUES (1, 'Mumbai', CURRENT_DATE, NULL, 'Y');

### Result
| customer_id | city | is_current |
|---|---|---|
| 1 | Mumbai | Y |